In [11]:
from one.api import ONE
from brainbox.population.decode import get_spike_counts_in_bins
from brainbox.io.one import SpikeSortingLoader, SessionLoader
from brainbox.ephys_plots import plot_brain_regions
from iblatlas.atlas import AllenAtlas
from brainwidemap import bwm_query, load_good_units, load_trials_and_mask, bwm_units
from brainwidemap.bwm_loading import merge_probes
from brainbox.behavior.training import (
    compute_performance,
    plot_psychometric,
    plot_reaction_time,
    get_signed_contrast,
    compute_reaction_time,
)
from brainbox.task.trials import find_trial_ids
from brainbox.io.one import SessionLoader
from pathlib import Path
from brainbox.task.trials import get_event_aligned_raster, get_psth
from brainbox.singlecell import bin_spikes2D
import numpy as np
from iblatlas.atlas import BrainRegions
from matplotlib import pyplot as plt
import seaborn as sns
import pandas as pd
import itertools
import pickle as pkl
from tqdm import tqdm
from pathlib import Path
import warnings
from sklearn.ensemble import RandomForestClassifier
import os
import concurrent.futures
import functools
from ibl_info.selective_decomposition import run_analysis_single_session, filter_eids
from ibl_info.utils import check_config

In [3]:
important_regions = [
    "VISp",
    "MOs",
    "SSp-ul",
    "ACAd",
    "PL",
    "CP",
    "VPM",
    "MG",
    "LGd",
    "ZI",
    "SNr",
    "MRN",
    "SCm",
    "PAG",
    "APN",
    "RN",
    "PPN",
    "PRNc",
    "PRNr",
    "GRN",
    "IRN",
    "PGRN",
    "CUL4 5",
    "SIM",
    "IP",
]

In [4]:
one = ONE(
    base_url="https://openalyx.internationalbrainlab.org",
    username="intbrainlab",
    password="international",
)

Connected to https://openalyx.internationalbrainlab.org as user "intbrainlab"


In [5]:
config = check_config()

In [6]:
unit_df = bwm_units(one)
all_tasks_to_run = []
for region in important_regions:
    selective_eids = filter_eids(unit_df, region, significant_filter=config["decoder_filter"])
    all_tasks_to_run.append(selective_eids)

Loading bwm_query results from fixtures/2023_12_bwm_release.csv
d16d0b38d392b18c0ce8b615ec89d60d7c901df2eeb3432986b62130af28ef01


In [7]:
all_tasks_to_run = np.concatenate(all_tasks_to_run)

In [8]:
eids_all = np.unique(all_tasks_to_run)

In [9]:
trials, mask = load_trials_and_mask(one=one, eid=eids_all[1], exclude_unbiased=True)

In [10]:
# for all animals
# compute performance
# look at proportion correct by contra

In [ ]:
performance, contrasts, n_contrasts = compute_performance(trials[mask])

(array([1.        , 0.97058824, 0.9       , 0.87272727, 0.55384615,
        0.43396226, 0.56097561, 0.76271186, 0.96428571]),
 array([-100.  ,  -25.  ,  -12.5 ,   -6.25,    0.  ,    6.25,   12.5 ,
          25.  ,  100.  ]),
 array([66, 68, 50, 55, 65, 53, 41, 59, 56]))

In [13]:
compute_reaction_time(trials=trials)

(array([0.27814432, 0.35362668, 0.47114283, 0.53802153, 0.56692932,
        0.65729418, 0.55739474, 0.5263143 , 0.37905278]),
 array([-100.  ,  -25.  ,  -12.5 ,   -6.25,    0.  ,    6.25,   12.5 ,
          25.  ,  100.  ]),
 array([111, 111,  95,  94, 107,  92,  78,  98,  81]))

In [14]:
compute_reaction_time(trials=trials[mask])

(array([0.29255438, 0.40144002, 0.53494438, 0.65783449, 0.60985235,
        0.69267529, 0.54410584, 0.4701094 , 0.38599935]),
 array([-100.  ,  -25.  ,  -12.5 ,   -6.25,    0.  ,    6.25,   12.5 ,
          25.  ,  100.  ]),
 array([66, 68, 50, 55, 65, 53, 41, 59, 56]))

In [22]:
df = trials[mask].copy()

In [23]:
df["signed"] = get_signed_contrast(df)

In [25]:
def get_congruency_performance(df):
    """
    Calculates the fraction of correct trials grouped by signed contrast
    and congruency (Congruent vs Incongruent).

    - Removes trials with 0.5 probability (neutral).
    - Includes 0 contrast trials by identifying their nominal side
      (based on which column is not NaN).
    """
    # 1. Filter out Neutral Blocks (Probability == 0.5)
    #    We use copy() to avoid SettingWithCopy warnings on the slice
    df_clean = df[df["probabilityLeft"] != 0.5].copy()

    # 2. Identify Nominal Side (Critical for 0 contrast inclusion)
    #    If contrastLeft is present (even if 0.0), it's a "Left Stimulus" trial.
    #    If contrastRight is present (even if 0.0), it's a "Right Stimulus" trial.
    has_left_stim = df_clean["contrastLeft"].notna()
    has_right_stim = df_clean["contrastRight"].notna()

    # 3. Calculate Signed Contrast
    #    Left is negative, Right is positive
    df_clean["signed_contrast"] = df_clean["contrastRight"].fillna(0) - df_clean[
        "contrastLeft"
    ].fillna(0)

    # 4. Define Congruency
    #    Congruent: Prior matches the Nominal Stimulus Side
    #    Incongruent: Prior mismatch Nominal Stimulus Side

    #    Logic:
    #    (Left Stim  AND High Left Prior) OR (Right Stim AND Low Left Prior)
    is_congruent = (has_left_stim & (df_clean["probabilityLeft"] > 0.5)) | (
        has_right_stim & (df_clean["probabilityLeft"] < 0.5)
    )

    df_clean["congruency"] = np.where(is_congruent, "Congruent", "Incongruent")

    # 5. Identify Correct Trials
    #    Assuming feedbackType 1.0 is correct
    df_clean["is_correct"] = (df_clean["feedbackType"] == 1.0).astype(int)

    # 6. Group and Aggregate
    summary = (
        df_clean.groupby(["signed_contrast", "congruency"])["is_correct"]
        .agg(
            accuracy="mean",
            n_trials="count",
            std_err=lambda x: x.std() / np.sqrt(len(x)),  # Standard Error
        )
        .reset_index()
    )

    return summary

90     100.00
91      -6.25
98    -100.00
99      -6.25
100   -100.00
        ...  
847    100.00
848      6.25
850      6.25
855      0.00
858      0.00
Name: signed, Length: 513, dtype: float64